# 1일차 6교시 — SARSA와 Q-Learning 소개

**PyTorch로 배우는 강화학습 · 1일차 Tabular-based Methods · 2026-07-27 (월)**

이애본 (Ph.D Aebon) · DreamIT Biz · https://pytorch26.dreamitbiz.com

---

## 🎯 학습목표

- TD 제어에서 SARSA와 Q-Learning의 업데이트 식을 구분한다
- On-policy와 Off-policy의 차이를 설명할 수 있다

---

# ⚡ 실행 방법 두 가지 — 편한 쪽을 고르세요

### 방법 ① 통째로 한 번에
바로 아래 **[통째로 실행]** 셀 **하나만** 실행하면 끝까지 돕니다.
결과부터 보고 싶으신 분께 권합니다.

### 방법 ② 단계별로 하나씩
그 아래 **[단계별]** 부분을 위에서부터 `Shift + Enter` 로 하나씩 실행하세요.
모두 **31칸**입니다. 한 칸 돌리고 결과 보고 넘어가면 됩니다.

> **이 교시는 혼자 돌아갑니다.** 앞 교시를 먼저 실행하지 않아도 됩니다.
> (앞 교시에서 만든 것을 이 노트북 안에 다시 넣어 뒀습니다 — 사이트의 *이 교시 전체 코드* 와 같은 판입니다.)
> 설치할 것도 없습니다 — 코랩에 다 들어 있습니다.

---

# ① 통째로 한 번에 실행

GitHub 에서 원본을 받아 그대로 돌립니다. 원본이 고쳐지면 자동으로 최신을 받습니다.

In [ ]:
!curl -sL https://raw.githubusercontent.com/aebonlee/pytorch26-lab/main/day1/standalone/06_update_rules.py -o 06_update_rules.py
!python 06_update_rules.py

---

# ② 단계별로 하나씩 실행

이 교시 내용이 **31칸**입니다.
위에서부터 `Shift + Enter`.

> ①을 이미 돌리셨어도 상관없습니다. 처음부터 다시 하는 것과 같습니다.

### 1 / 31 칸

In [ ]:
# ============================================================
# 1일차 6교시 — SARSA와 Q-Learning 소개
# 복사해서 그대로 실행하면 됩니다. 고칠 것 없습니다.
# ------------------------------------------------------------
# 이 교시 코드는 앞 교시의 변수·클래스를 이어 씁니다.
# 그래서 이 블록에는 **여기까지 필요한 코드가 전부** 들어 있습니다.
# (수업용 코드만 따로 복사하면 NameError 가 납니다 — 그건 정상입니다.)
# ============================================================

# ── 1교시에서 이어받음 — 강화학습 소개 ──
import numpy as np                          # 숫자 계산 도구 (파이썬의 계산기)

### 2 / 31 칸

In [ ]:
# ============================================================
# 슬롯머신 10대 중에 좋은 걸 찾기 — 탐험과 활용의 첫 만남
# ------------------------------------------------------------
# 슬롯머신이 10대 있습니다. 각각 나오는 돈의 평균이 다릅니다.
# 그런데 어느 게 좋은지는 ==해봐야 알 수 있습니다.==
#
# 여기서 딜레마가 생깁니다.
#   활용(exploit) : 지금까지 제일 좋았던 걸 계속 당긴다
#   탐험(explore) : 다른 것도 가끔 당겨 본다
#
# 활용만 하면? 처음 운 좋게 나온 것에 갇힙니다.
# 탐험만 하면? 좋은 걸 알면서도 계속 딴 걸 당깁니다.
# ============================================================

np.random.seed(0)                           # 결과를 항상 같게 만든다 (수업용)

n_arms = 10                                 # 슬롯머신 10대
true_means = np.random.normal(0, 1, n_arms) # 각 기계의 '진짜' 평균 (우리는 모른다고 치자)

### 3 / 31 칸

In [ ]:
def run_bandit(epsilon, steps=2000):
    """
    epsilon 확률로 아무거나 당기고, 나머지는 제일 좋아 보이는 걸 당긴다.
    steps 번 당겨 보고 평균 수익을 돌려준다.
    """
    Q = np.zeros(n_arms)        # 각 기계가 얼마나 좋은지 내 '추정치'. 0에서 시작.
    N = np.zeros(n_arms)        # 각 기계를 몇 번 당겼는지 세는 통

    rewards = []                # 매번 받은 돈을 기록

    for t in range(steps):      # steps 번 반복
        # ── ① 어느 기계를 당길지 고른다 ──
        if np.random.rand() < epsilon:      # 0~1 사이 무작위 수가 epsilon 보다 작으면
            a = np.random.randint(n_arms)   #   아무거나 고른다 (탐험)
        else:
            a = np.argmax(Q)                #   추정치가 가장 큰 걸 고른다 (활용)

        # ── ② 실제로 당겨 본다 ──
        r = np.random.normal(true_means[a], 1)   # 진짜 평균 근처에서 값이 나온다
                                                 # 1은 흔들림의 크기 (매번 다르게 나온다)

        # ── ③ 결과를 반영해 추정치를 고친다 ──
        N[a] += 1                            # 이 기계를 한 번 더 당겼다고 기록
        Q[a] += (r - Q[a]) / N[a]            # 추정치를 새 결과 쪽으로 조금 옮긴다
        # 이 한 줄이 '평균 구하기'입니다. 다 모아 뒀다 나누지 않고
        # 나올 때마다 조금씩 옮겨 가는 방식입니다. 강화학습 내내 이 모양이 나옵니다.
        #   새 추정 = 옛 추정 + (실제로 나온 것 - 옛 추정) x 얼마나 반영할지

        rewards.append(r)                    # 받은 돈 기록

    return np.mean(rewards)                  # 평균 수익을 돌려준다

### 4 / 31 칸

In [ ]:
# ── epsilon 을 바꿔 가며 비교 ────────────────────────────
print('epsilon = 아무거나 당겨 볼 확률')
print()

### 5 / 31 칸

In [ ]:
for eps in [0.0, 0.01, 0.1, 0.5]:
    print(f"epsilon={eps:4.2f}  평균 보상 = {run_bandit(eps):.3f}")

print("""
결과 읽는 법
  epsilon = 0.00  탐험을 아예 안 합니다.
                  처음 운 좋게 나온 기계에 갇혀서 더 좋은 걸 영영 못 찾습니다.
  epsilon = 0.01  아주 가끔만 둘러봅니다. 조심스럽습니다.
  epsilon = 0.10  보통 쓰는 값. 대개 여기쯤이 가장 좋습니다.
  epsilon = 0.50  절반을 딴 데 씁니다. 좋은 걸 알면서도 낭비합니다.

→ 너무 안 해봐도 안 되고, 너무 많이 해봐도 안 됩니다.
  이 균형이 강화학습 3일 내내 따라다니는 문제입니다.

  1일차 : 사람이 epsilon 을 정해 준다
  2일차 : 처음엔 크게 나중엔 작게 줄여 나간다
  3일차 : 아예 목표에 넣어서 스스로 조절하게 만든다
""")

### 6 / 31 칸

In [ ]:
# ============================================================
# 바꿔 보기
#   1) steps 를 200 으로 줄이면? → 탐험할 시간이 부족해 결과가 나빠집니다
#   2) steps 를 20000 으로 늘리면? → epsilon 이 작아도 결국 좋은 걸 찾습니다
#   3) np.random.seed(0) 의 0을 1, 2 로 바꿔 보세요.
#      순위가 바뀔 수도 있습니다 — 한 번의 결과로 판단하면 안 된다는 뜻입니다.
# ============================================================

# ── 2교시에서 이어받음 — MDP 소개 ──
import numpy as np                          # 숫자 계산 도구

### 7 / 31 칸

In [ ]:
# ============================================================
# 4x4 격자 세상 만들기 — 오늘 하루 계속 쓸 놀이판
# ------------------------------------------------------------
# 강화학습 문제를 적는 표준 양식이 MDP 입니다. 다섯 가지만 정하면 됩니다.
#   상태(s)  지금 어느 칸에 있나
#   행동(a)  어느 방향으로 갈까
#   보상(r)  그 행동으로 몇 점 받았나
#   전이     그 행동을 하면 어느 칸이 되나
#   감마(γ)  미래를 얼마나 챙길지
#
# 이 파일에서 만드는 놀이판을 3·4교시에서 계속 씁니다.
# ============================================================

N = 4                                       # 4줄 4칸짜리 격자
n_states = N * N                            # 칸이 모두 16개 (0번부터 15번까지)
n_actions = 4                               # 행동 4가지 (상·하·좌·우)

TERMINALS = [0, n_states - 1]               # 0번 칸과 15번 칸에 도착하면 끝
                                            # (왼쪽 위 구석과 오른쪽 아래 구석)

# 칸 번호는 이렇게 매겨져 있습니다:
#    0  1  2  3
#    4  5  6  7
#    8  9 10 11
#   12 13 14 15

### 8 / 31 칸

In [ ]:
def step(s, a):
    """
    s번 칸에서 a 방향으로 가면 어떻게 되는지 알려 주는 함수.
    돌려주는 값: (다음 칸 번호, 받은 점수)

    이 격자는 '결정적'입니다 — 오른쪽으로 가면 반드시 오른쪽으로 갑니다.
    (미끄러지는 얼음판 같은 건 나중에 다룹니다)
    """
    if s in TERMINALS:                      # 이미 끝난 칸이면
        return s, 0                         # 그 자리에 있고 점수도 없다

    r, c = divmod(s, N)                     # 칸 번호를 (몇 번째 줄, 몇 번째 칸)으로 바꾼다
                                            # 예: 5번 칸 → divmod(5,4) = (1, 1) → 1줄 1칸

    if a == 0:   r = max(r - 1, 0)          # 위로  (0번 줄보다 위로는 못 감 → 벽)
    elif a == 1: r = min(r + 1, N - 1)      # 아래로 (마지막 줄보다 아래로는 못 감)
    elif a == 2: c = max(c - 1, 0)          # 왼쪽으로
    elif a == 3: c = min(c + 1, N - 1)      # 오른쪽으로
    # max/min 을 쓰는 이유: 벽에 부딪히면 제자리에 있게 하려고

    return r * N + c, -1                    # (줄, 칸)을 다시 칸 번호로 바꾸고 점수 -1
    # 왜 점수가 -1 일까요?
    #   한 걸음 걸을 때마다 -1점을 받습니다. 즉 ==빨리 끝낼수록 손해가 적습니다.==
    #   "최대한 빨리 목표까지 가라"를 점수로 표현한 것입니다.

### 9 / 31 칸

In [ ]:
# ── 전이표를 미리 만들어 둔다 ────────────────────────────
# P[s][a] 하면 바로 (다음 칸, 점수)가 나옵니다.
# 매번 step() 을 부르는 것보다 빠르고, 3·4교시 코드가 짧아집니다.
P = [[step(s, a) for a in range(n_actions)] for s in range(n_states)]
# 위 한 줄은 이중 반복문입니다. 풀어 쓰면:
#   P = []
#   for s in range(n_states):
#       row = []
#       for a in range(n_actions):
#           row.append(step(s, a))
#       P.append(row)

print('전이표 P 를 만들었습니다.')
print('  P[5][3] =', P[5][3], ' ← 5번 칸에서 오른쪽으로 가면 6번 칸, 점수 -1')
print('  P[0][3] =', P[0][3], ' ← 0번 칸은 끝난 칸이라 그대로, 점수 0')
print()

### 10 / 31 칸

In [ ]:
# ── 아무렇게나 걸어 보기 ─────────────────────────────────
# 정책(policy)이란 "이 칸에서는 이렇게 한다"는 규칙입니다.
# 지금은 아무 규칙 없이 무작위로 걸어 봅니다.

s = 5                                       # 5번 칸에서 출발
trajectory = []                             # 걸어간 기록을 담을 곳

rng = np.random.default_rng(42)             # 무작위 생성기 (42는 결과 고정용 숫자)

### 11 / 31 칸

In [ ]:
while s not in TERMINALS:                   # 끝나는 칸에 닿을 때까지 반복
    a = rng.integers(n_actions)             # 0~3 중 아무거나 (무작위 정책)
    s_next, r = P[s][a]                     # 그 방향으로 가면 어떻게 되나
    trajectory.append((s, a, r))            # 기록: (어디서, 뭘 했고, 몇 점)
    s = s_next                              # 다음 칸으로 이동

print(f"무작위로 걸었더니")
print(f"  걸음 수  : {len(trajectory)}")
print(f"  총 점수  : {sum(t[2] for t in trajectory)}")
print(f"""
  → 한 걸음마다 -1점이니 ==걸음 수와 총 점수는 부호만 다릅니다.==
    아무렇게나 걸으면 한참 헤맵니다.

  다음 시간부터 "어떻게 하면 덜 헤맬까"를 계산으로 찾습니다.
""")

### 12 / 31 칸

In [ ]:
# ============================================================
# 바꿔 보기
#   1) 출발 칸 s = 5 를 다른 번호로 바꿔 보세요.
#      끝나는 칸(0, 15)에 가까울수록 빨리 끝납니다.
#   2) rng 의 42 를 1, 2, 3 으로 바꿔 여러 번 돌려 보세요.
#      ==같은 무작위 정책인데도 걸음 수가 크게 달라집니다.==
#      이 흔들림이 강화학습을 어렵게 만드는 요소 중 하나입니다.
#   3) 벽에 부딪히면 제자리에 있는 것을 확인해 보세요.
#      print(P[0][0]) 을 찍어 보시면 됩니다 (0번 칸에서 위로 = 제자리).
# ============================================================

# ── 3교시에서 이어받음 — Dynamic Programming 소개 ──
import numpy as np                          # 숫자 계산 도구

### 13 / 31 칸

In [ ]:
# ============================================================
# 정책 평가 — "지금 이렇게 하면 각 칸이 얼마나 좋은가" 계산하기
# ------------------------------------------------------------
# 2교시에서 만든 놀이판(P, n_states, TERMINALS)을 그대로 씁니다.
#
# 오늘은 규칙을 전부 알고 있는 경우입니다. 지도를 다 갖고 있는 셈이죠.
# 그러면 ==직접 걸어 보지 않고 계산만으로== 답을 낼 수 있습니다.
#
# 계산의 핵심이 벨만 방정식입니다.
#   여기가 얼마나 좋은가 = 지금 받는 점수 + 감마 x 다음 칸이 얼마나 좋은가
# ============================================================

gamma = 1.0                 # 미래를 얼마나 챙길지. 1.0 = 지금과 똑같이 챙긴다.
theta = 1e-6                # 값이 이만큼도 안 변하면 "다 됐다"고 본다

V = np.zeros(n_states)      # 각 칸의 값. 전부 0에서 시작합니다.
                            # 처음엔 엉터리지만 반복하면 맞아 들어갑니다.

iteration = 0               # 몇 번 반복했는지 세는 통

### 14 / 31 칸

In [ ]:
while True:                 # 값이 더 안 변할 때까지 반복
    delta = 0.0             # 이번 바퀴에서 값이 가장 많이 바뀐 정도
    V_new = V.copy()        # 새 값을 담을 곳 (원본을 놔두고 따로 계산)

    for s in range(n_states):          # 모든 칸을 하나씩 돌면서
        if s in TERMINALS:             # 끝나는 칸은
            continue                   #   계산할 것이 없으니 건너뛴다

        # ── 이 칸의 값을 새로 계산한다 ──
        # 지금 정책은 "아무 방향이나 똑같은 확률로" 입니다.
        # 4방향이니 각각 0.25 확률입니다.
        v = 0.0
        for a in range(n_actions):     # 4방향을 하나씩
            s_next, r = P[s][a]        # 그 방향으로 가면 어디로 가고 몇 점인가
            v += 0.25 * (r + gamma * V[s_next])
            #     ^^^^ 그 방향을 고를 확률
            #            ^^^^^^^^^^^^^^^^^^^^^^ 벨만 방정식 그 자체
            #            지금 점수 + 감마 x 다음 칸의 값

        V_new[s] = v                   # 새로 계산한 값을 넣는다
        delta = max(delta, abs(v - V[s]))   # 얼마나 바뀌었는지 기록

    V = V_new                          # 한 바퀴 끝났으니 값을 교체
    iteration += 1

    if delta < theta:                  # 거의 안 변했으면
        break                          #   끝난 것으로 본다


print(f"{iteration}회 반복 후 수렴했습니다")
print()
print(np.round(V.reshape(4, 4), 1))    # 4x4 표 모양으로 보기 좋게 출력
print(f"""
결과 읽는 법
  0번 칸(왼쪽 위)과 15번 칸(오른쪽 아래)은 끝나는 칸이라 0 입니다.
  ==끝나는 칸에서 멀수록 값이 낮습니다(-20 근처).==
  거기까지 가는 데 걸음이 많이 필요하고, 한 걸음마다 -1점이니까요.

  이 값은 "아무렇게나 걸었을 때"의 값입니다.
  다음 시간에는 "제일 좋은 길로 걸으면 어떻게 되는지"를 봅니다.
""")

### 15 / 31 칸

In [ ]:
# ============================================================
# 여기서 꼭 짚고 갈 것
#
#   처음에 V 를 전부 0으로 두고 시작했습니다. 명백히 엉터리 값이죠.
#   그런데도 반복하니 맞아 들어갔습니다. 왜 그럴까요?
#
#   매 바퀴마다 "실제 점수 r" 이 조금씩 섞여 들어가기 때문입니다.
#   끝나는 칸 근처부터 값이 정해지고, 그게 한 칸씩 뒤로 번져 나갑니다.
#   ==값이 목표에서부터 뒤로 퍼지는 것== — 이게 강화학습의 기본 그림입니다.
#
# 바꿔 보기
#   1) gamma 를 0.9 로 낮추면? → 값이 전체적으로 덜 낮아집니다
#      (먼 미래의 -1 이 덜 반영되니까요)
#   2) gamma 를 0.5 로 낮추면? → 지형이 훨씬 평평해집니다
#   3) 0.25 를 [0.4, 0.1, 0.1, 0.4] 처럼 바꿔 보세요.
#      "위·오른쪽을 더 자주 가는 정책"의 값이 나옵니다.
#   4) while 문 안에 print(np.round(V.reshape(4,4),1)) 을 넣어 보세요.
#      값이 한 바퀴마다 어떻게 번져 나가는지 눈으로 볼 수 있습니다.
# ============================================================

# ── 4교시에서 이어받음 — Policy Iteration, Value Iteration 구현 ──
import numpy as np                          # 숫자 계산 도구

### 16 / 31 칸

In [ ]:
# ============================================================
# 최적 정책 찾기 — 두 가지 방법을 나란히
# ------------------------------------------------------------
# 3교시에서는 "아무렇게나 걸었을 때" 각 칸의 값을 구했습니다.
# 이제 ==제일 좋은 길을 찾습니다.==
#
# 방법이 두 가지입니다. 성격이 다릅니다.
#   정책 반복 : 신중한 사람. 끝까지 계산하고 나서 방식을 고친다.
#   가치 반복 : 성급한 사람. 계산이 덜 끝나도 일단 좋아 보이는 쪽으로 간다.
#
# 둘 다 결국 같은 답에 도착합니다. 가는 방식만 다릅니다.
# ============================================================

gamma = 1.0                 # 미래를 지금과 똑같이 챙긴다

### 17 / 31 칸

In [ ]:
def q_from_v(V, s):
    """
    s번 칸에서 각 방향으로 갔을 때의 값(Q)을 계산한다.
    돌려주는 것: 숫자 4개 (상·하·좌·우 각각의 값)

    Q = 그 방향으로 갔을 때 받는 점수 + 감마 x 도착한 칸의 값
    """
    return np.array([P[s][a][1] + gamma * V[P[s][a][0]]
                     for a in range(n_actions)])
    #                P[s][a][1] = 점수,  P[s][a][0] = 다음 칸 번호


# ============================================================
# 방법 ① 정책 반복 (Policy Iteration) — 신중한 사람
# ============================================================

### 18 / 31 칸

In [ ]:
def policy_iteration():
    policy = np.zeros(n_states, dtype=int)   # 처음엔 모든 칸에서 '위쪽(0번)'으로 간다
                                             # 아무렇게나 정해도 됩니다. 어차피 고쳐 나갈 거니까요.

    while True:                              # 더 고칠 게 없을 때까지 반복

        # ── 1단계: 평가 — 지금 방식대로 하면 각 칸이 얼마나 좋은가 ──
        #
        # ★ MAX_SWEEP 이 왜 필요한가 (실제로 겪은 문제입니다) ★
        #   처음 정책은 모든 칸에서 '위쪽'입니다.
        #   맨 윗줄에서는 위로 가려다 벽에 막혀 제자리에 머뭅니다.
        #   그러면 V[s] = -1 + 1.0 x V[s] 가 되어 값이 끝없이 내려갑니다.
        #   gamma 가 1.0 이라 줄어들지도 않습니다.
        #   → delta < 1e-6 조건이 영원히 안 맞아서 무한 루프에 빠집니다.
        #
        #   그래서 "1000바퀴만 돌고 다음 단계로 넘어간다"고 상한을 둡니다.
        #   다음 정책은 끝나는 칸에 도달하게 되므로 이후로는 정상 수렴합니다.
        #   이런 방식을 modified policy iteration 이라고 부릅니다.
        MAX_SWEEP = 1000

        V = np.zeros(n_states)               # 값을 0에서 다시 시작

        for _ in range(MAX_SWEEP):           # 최대 1000바퀴
            delta = 0.0
            for s in range(n_states):
                if s in TERMINALS: continue  # 끝나는 칸은 건너뛴다

                s_next, r = P[s][policy[s]]  # 지금 정책이 시키는 방향으로 간다
                v = r + gamma * V[s_next]    # 벨만 방정식 (3교시와 같은 식)

                delta = max(delta, abs(v - V[s]))
                V[s] = v                     # 바로 덮어쓴다 (같은 바퀴 안에서도 반영)

            if delta < 1e-6: break           # 거의 안 변하면 평가 끝

        # ── 2단계: 개선 — 더 좋은 방향이 있으면 바꾼다 ──
        stable = True                        # "아무것도 안 바뀌었다"고 일단 가정

        for s in range(n_states):
            if s in TERMINALS: continue

            best_a = np.argmax(q_from_v(V, s))   # 값이 가장 큰 방향을 찾는다

            if best_a != policy[s]:          # 지금 방향과 다르면
                stable = False               #   바뀐 것이 있다고 표시하고
                policy[s] = best_a           #   그 방향으로 바꾼다

        if stable:                           # 한 칸도 안 바뀌었으면
            return policy, V                 #   최적 정책을 찾은 것

### 19 / 31 칸

In [ ]:
# ============================================================
# 방법 ② 가치 반복 (Value Iteration) — 성급한 사람
# ============================================================
def value_iteration():
    V = np.zeros(n_states)

    while True:
        delta = 0.0
        for s in range(n_states):
            if s in TERMINALS: continue

            v = q_from_v(V, s).max()         # ★ 여기가 핵심 ★
            # 정책 반복은 "지금 정책이 시키는 방향"의 값을 썼습니다.
            # 가치 반복은 그냥 ==가장 좋은 방향의 값==을 씁니다.
            # 평가와 개선을 한 줄에 합쳐 버린 것입니다.

            delta = max(delta, abs(v - V[s]))
            V[s] = v

        if delta < 1e-6: break               # 값이 거의 안 변하면 끝

    # 값이 다 정해진 뒤에 정책을 한 번만 뽑아낸다
    policy = np.array([np.argmax(q_from_v(V, s)) for s in range(n_states)])
    return policy, V

### 20 / 31 칸

In [ ]:
# ── 두 방법을 돌려서 비교 ────────────────────────────────
arrows = np.array(['↑', '↓', '←', '→'])      # 숫자 0,1,2,3 을 화살표로 보여 주려고

pi_policy, pi_V = policy_iteration()          # 정책 반복
vi_policy, vi_V = value_iteration()           # 가치 반복

print("정책 반복이 찾은 최적 정책:")
print(arrows[pi_policy].reshape(4, 4))        # 화살표를 4x4 로 배열
print()
print("가치 반복이 찾은 최적 정책:")
print(arrows[vi_policy].reshape(4, 4))
print()
print("두 방법의 가치함수가 같은가:", np.allclose(pi_V, vi_V))
print(f"""
결과 읽는 법
  각 칸의 화살표는 ==그 칸에서 가장 좋은 방향==입니다.
  전부 끝나는 칸(왼쪽 위 또는 오른쪽 아래) 쪽을 가리키면 제대로 된 것입니다.

  구석 칸의 화살표는 둘 다 맞는 경우가 있어 방법마다 다를 수 있습니다.
  거리가 같으면 어느 쪽으로 가도 똑같기 때문입니다.
  ==중요한 건 화살표가 아니라 값이 같다는 것입니다.== 마지막 줄이 True 면 맞습니다.

  → 신중하게 가나 성급하게 가나 도착점은 같습니다.
    실무에서는 코드가 짧은 가치 반복을 더 많이 씁니다.
""")

### 21 / 31 칸

In [ ]:
# ============================================================
# 바꿔 보기
#   1) gamma 를 0.9 로 낮추고 값을 출력해 보세요 (print(np.round(vi_V.reshape(4,4),1)))
#      → 지형이 덜 가파릅니다. 먼 미래를 덜 챙기니까요.
#   2) gamma 를 0.5 로 낮추면? → 거의 평평해집니다.
#      멀리 있는 목표가 지금 칸의 값에 거의 영향을 안 줍니다.
#   3) 가치 반복이 몇 바퀴 만에 끝나는지 세어 보세요.
#      while 문 안에 카운터를 하나 두시면 됩니다.
#   4) MAX_SWEEP 을 5 로 줄여 보세요.
#      평가를 대충 해도 결국 같은 답에 도착하는 것을 볼 수 있습니다.
#      (그래서 modified policy iteration 이 실용적입니다)
# ============================================================

# ── 5교시에서 이어받음 — Monte-Carlo 방법, Temporal Difference 방법 소개 ──
import numpy as np                      # 숫자 계산을 도와주는 도구

# 같은 문제를 두 가지 방법으로 풀어 보고 결과를 비교합니다.
#   방법 1) 몬테카를로 — 한 판을 끝까지 하고 나서 배우기
#   방법 2) 시간차(TD)  — 한 걸음 옮길 때마다 바로 배우기

rng = np.random.default_rng(0)         # 무작위 뽑기 도구. 0은 '항상 같은 결과'를 위한 값
alpha = 0.05                           # 학습률 — 새로 안 것을 얼마나 믿을지 (0~1)
gamma = 1.0                            # 미래를 얼마나 챙길지 (1이면 먼 미래도 그대로)

### 22 / 31 칸

In [ ]:
def gen_episode():
    """한 판을 끝까지 해보고, 지나온 기록을 돌려줍니다."""
    s = rng.integers(1, n_states - 1)  # 출발 칸을 아무 데나 고름 (끝 칸 제외)
    episode = []                       # 지나온 기록을 담을 빈 목록

    while s not in TERMINALS:          # 끝 칸에 도착할 때까지 반복
        a = rng.integers(n_actions)    # 아무 방향이나 하나 고름 (무작위로 걷기)
        s_next, r = P[s][a]            # 그 방향으로 가면 어디로 가고 몇 점인지
        episode.append((s, r))         # "어느 칸에서 몇 점 받았다"를 기록
        s = s_next                     # 다음 칸으로 이동

    return episode                     # 한 판의 기록 전체를 돌려줌

### 23 / 31 칸

In [ ]:
# ── 방법 1) 몬테카를로 — 끝까지 하고 나서 배우기 ──────────
V_mc = np.zeros(n_states)              # 각 칸의 값. 처음엔 전부 0 (아무것도 모름)

### 24 / 31 칸

In [ ]:
for _ in range(5000):                  # 5000판 반복
    episode = gen_episode()            # 한 판을 끝까지 해봄
    G = 0.0                            # 이 지점부터 끝까지 받은 총 점수
    visited = set()                    # 이번 판에서 이미 들른 칸 목록

    # 뒤에서부터 되짚습니다. 끝에서부터 세야 "여기부터 끝까지"가 계산됩니다.
    for s, r in reversed(episode):
        G = r + gamma * G              # (지금 점수) + (여기 다음부터 끝까지)

        if s not in visited:           # 같은 칸을 여러 번 지났으면 첫 번째만 사용
            visited.add(s)             # 들렀다고 표시
            # 지금 알던 값(V_mc[s])을 실제 결과(G) 쪽으로 조금(alpha) 옮깁니다.
            V_mc[s] += alpha * (G - V_mc[s])

### 25 / 31 칸

In [ ]:
# ── 방법 2) 시간차(TD) — 한 걸음마다 바로 배우기 ──────────
V_td = np.zeros(n_states)              # 마찬가지로 전부 0에서 시작

### 26 / 31 칸

In [ ]:
for _ in range(5000):                  # 5000판 반복
    s = rng.integers(1, n_states - 1)  # 아무 칸에서 출발

    while s not in TERMINALS:          # 끝 칸에 닿을 때까지
        a = rng.integers(n_actions)    # 아무 방향이나 고름
        s_next, r = P[s][a]            # 가보니 어디로 갔고 몇 점인지

        # 여기가 몬테카를로와 다른 곳입니다.
        # 끝까지 안 가고, "지금 점수 + 다음 칸의 (아직 부정확한) 값"으로 대신합니다.
        td_error = r + gamma * V_td[s_next] - V_td[s]

        V_td[s] += alpha * td_error    # 그 차이만큼 조금 옮김
        s = s_next                     # 다음 칸으로
        s = s_next

print("MC 추정:"); print(np.round(V_mc.reshape(4, 4), 1))
print("TD 추정:"); print(np.round(V_td.reshape(4, 4), 1))
# 둘 다 DP 정답(-14, -20, -22...)에 근접하는지 확인하세요

### 27 / 31 칸

In [ ]:
# ── 오늘 이 교시 — SARSA와 Q-Learning 소개 ──
# 두 알고리즘의 차이는 단 한 줄 — TD 목표(target)의 정의

# Q 는 '표'입니다. Q[칸][방향] = 그 칸에서 그 방향으로 가면 얼마나 좋은지.
# 아래 두 함수는 그 표의 숫자 하나를 고치는 방법입니다.
# 두 함수는 딱 한 줄만 다릅니다. 그 줄을 잘 보세요.


# ── 방법 1) SARSA — 내가 "실제로 할" 행동을 보고 고침 ──────
def sarsa_update(Q, s, a, r, s_next, a_next, alpha=0.1, gamma=0.99):
    # s      : 지금 있는 칸
    # a      : 지금 하려는 행동
    # r      : 그 행동으로 받은 점수
    # s_next : 그래서 가게 된 다음 칸
    # a_next : 다음 칸에서 "실제로 할" 행동   ← SARSA 는 이게 필요합니다
    # alpha  : 얼마나 믿을지 (0.1이면 10%만 반영)
    # gamma  : 미래를 얼마나 챙길지

    # 목표값 = 지금 받은 점수 + 다음 칸에서 실제로 할 행동의 값
    target = r + gamma * Q[s_next][a_next]

    # 지금 알던 값을 목표값 쪽으로 alpha 만큼 조금 옮깁니다.
    # (목표값 - 지금값) 이 '내 예상이 얼마나 빗나갔나' 입니다.
    Q[s][a] += alpha * (target - Q[s][a])

### 28 / 31 칸

In [ ]:
# ── 방법 2) Q-러닝 — "제일 좋은" 행동을 보고 고침 ─────────
def q_learning_update(Q, s, a, r, s_next, alpha=0.1, gamma=0.99):
    # 여기는 a_next 가 없습니다. 다음에 뭘 할지 몰라도 되기 때문입니다.

    # 목표값 = 지금 받은 점수 + 다음 칸에서 "가장 좋은" 행동의 값
    # max 는 여러 값 중 제일 큰 것을 고르는 것입니다.
    # 실제로 그 행동을 할지는 상관하지 않습니다. "만약 최선을 다한다면" 을 가정합니다.
    target = r + gamma * max(Q[s_next])

    Q[s][a] += alpha * (target - Q[s][a])    # 고치는 방식은 위와 똑같습니다

### 29 / 31 칸

In [ ]:
# ── 행동 고르기 — 두 방법이 공통으로 씁니다 ───────────────
import random                                # 무작위 뽑기 도구

### 30 / 31 칸

In [ ]:
def epsilon_greedy(Q, s, n_actions, epsilon=0.1):
    # epsilon 은 '아무거나 해볼 확률' 입니다. 0.1이면 10번에 1번.
    # 왜 일부러 아무거나 할까요?
    #   처음 우연히 괜찮았던 길만 계속 가면 더 좋은 길을 영영 못 찾기 때문입니다.

    if random.random() < epsilon:            # 0~1 사이 아무 숫자를 뽑아서
        return random.randrange(n_actions)   # epsilon 보다 작으면 → 아무 방향이나

    # 그렇지 않으면 → 지금까지 알기로 가장 좋은 방향
    # key=... 는 "이 기준으로 가장 큰 것을 고르라" 는 뜻입니다.
    return max(range(n_actions), key=lambda a: Q[s][a])

### 31 / 31 칸

In [ ]:
# ============================================================
# 잘 만들어졌는지 확인 (이 부분이 있어야 실행했을 때 결과가 보입니다)
# ============================================================
print('두 갱신식이 어떻게 다른지 숫자로 확인합니다.')
print()

Q_demo = {0: [1.0, 5.0], 1: [3.0, 2.0]}      # 상태 2개, 행동 2개짜리 작은 표
r, gamma, alpha = 1.0, 0.9, 0.1
s, a, s2 = 0, 0, 1

a2_greedy = int(np.argmax(Q_demo[s2]))        # 가장 좋은 행동 (Q-러닝이 쓰는 것)
a2_actual = 1                                 # 실제로 하게 될 행동 (SARSA가 쓰는 것)

sarsa = Q_demo[s][a] + alpha * (r + gamma * Q_demo[s2][a2_actual] - Q_demo[s][a])
qlear = Q_demo[s][a] + alpha * (r + gamma * Q_demo[s2][a2_greedy] - Q_demo[s][a])

print(f'  다음 상태의 Q값 : {Q_demo[s2]}')
print(f'  SARSA  (실제 행동 {a2_actual}번 = {Q_demo[s2][a2_actual]}) -> 갱신값 {sarsa:.4f}')
print(f'  Q러닝  (최선 행동 {a2_greedy}번 = {Q_demo[s2][a2_greedy]}) -> 갱신값 {qlear:.4f}')
print()
print('  -> 같은 경험인데 값이 다릅니다. 딱 이 한 곳이 두 방법을 가릅니다.')

---

## 막히면

- 사이트의 같은 교시를 보세요 — 실행 결과와 해설이 그대로 있습니다.
  https://pytorch26.dreamitbiz.com/#/day/1/6
- 오류가 나면 **[막힐 때]** 메뉴부터.
  https://pytorch26.dreamitbiz.com/#/help

---

*Ph.D Aebon & Claude Code 협작 전자출판 도서 · © 2026 DreamIT Biz*